# QLoRA — Quantized Low-Rank Adaptation

A refresher on **QLoRA**: fine-tuning a large model on a single GPU by **freezing a 4-bit-quantized base
model** and training only small **LoRA adapters** on top of it. It's the technique that put 65B-parameter
fine-tuning on one 48 GB card and 7B on a free Colab T4. QLoRA = **Q** (4-bit NF4 quantization of the frozen
base) + **LoRA** (low-rank trainable adapters) + two memory tricks (double quantization, paged optimizers).

**Domain:** LLM Inference, Training & Optimization  ·  **recommended addition**  ·  **runnable:** yes  ·  _cross-ref [Quantization: GPTQ/AWQ/bitsandbytes](./quantization-gptq-awq.ipynb), [LoRA / ControlNet](./lora-controlnet.ipynb), [Unsloth](./unsloth.ipynb)_

## 1. What & Why

Full fine-tuning of an LLM updates **every** weight, so you must hold in GPU memory: the weights (fp16),
their gradients (fp16), and the optimizer state (Adam keeps two fp32 moments per weight). That's roughly
**16 bytes per parameter** — a 7B model needs ~112 GB before you've stored a single activation. Far out of
reach for one consumer GPU.

QLoRA collapses this to fit on one card with three moves:

1. **Quantize the base to 4-bit (NF4) and freeze it.** The frozen weights drop from 2 bytes to ~0.5 bytes
   each (~4× smaller, ~3.5 GB for 7B) and — crucially — they have **no gradients and no optimizer state**
   because they never update.
2. **Train tiny LoRA adapters in bf16.** Instead of updating `W`, you learn a low-rank correction
   `ΔW = B·A` (with rank `r`≈8–64) added alongside the frozen `W`. The adapters are a fraction of a percent
   of the parameters, so gradients + Adam state are negligible.
3. **Keep accuracy with NF4 + double quantization + paged optimizers** (details in §3). The headline result
   from the paper: 4-bit QLoRA fine-tuning matches 16-bit full fine-tuning quality.

**Reach for it when:** you want to fine-tune a model that doesn't fit for full training on your hardware —
which is almost always. **Skip it when:** the model already fits comfortably for full fine-tuning and you
need the absolute last bit of quality, or you only need *inference* (then just quantize for serving — see
the Quantization notebook — no adapters needed).

## 2. Mental Model

Picture the base model as a **read-only reference book compressed to 4 bits** — you can look things up
(forward pass) but you may not write in it. Learning happens on **sticky notes (LoRA adapters)** you attach
beside specific pages; at read time you combine page + sticky note. The book is huge but frozen and tiny per
character; the sticky notes are what you actually edit and save.

```
        x ──►┌─────────────────────┐
             │  frozen 4-bit base W │──► W·x          (no grad, no optimizer state)
             └─────────────────────┘        +
        x ──►┌──────────┐   ┌──────────┐          ──► output
             │  A (r×d)  │──►│  B (d×r)  │──► (α/r)·B·A·x   (bf16, trainable)
             └──────────┘   └──────────┘
                  ▲ down-project    ▲ up-project
              r ≪ d  ⇒  B·A is a rank-r correction to W
```

Two ideas make it click:

- **Low rank is enough.** The *update* a task needs (`ΔW`) lives in a tiny subspace, so `ΔW = B·A` with
  rank `r`≈8–64 captures it using `r·(d_in+d_out)` numbers instead of `d_in·d_out`. `A` starts random, `B`
  starts at **zero**, so the adapter is a no-op at step 0 — training begins exactly at the base model.
- **You quantize the part you freeze, and keep precision on the part you train.** The big frozen matrix can
  tolerate 4-bit noise (it's only ever read); the small adapter that carries the learning signal stays bf16.

One sentence: **QLoRA fine-tunes a 4-bit frozen model by learning a small high-precision low-rank patch,
so the expensive gradient/optimizer memory only ever touches a fraction of a percent of the weights.**

## 3. Key Concepts

| Term | What it means |
|------|---------------|
| **LoRA** | Low-Rank Adaptation. Freeze `W`; learn `ΔW = (α/r)·B·A` with `A∈ℝ^{r×d_in}`, `B∈ℝ^{d_out×r}`, rank `r` small. Only `A,B` train. |
| **Rank `r`** | Size of the bottleneck. Bigger `r` = more capacity + more trainable params. 8–16 is typical; 64 for harder tasks. |
| **`alpha` (scaling)** | LoRA output is scaled by `α/r`. A knob on the adapter's strength; people often set `α = 2r` and tune `r`. |
| **NF4 (NormalFloat4)** | QLoRA's 4-bit data type: its 16 levels sit at the **quantiles of a normal distribution**, matching the ~Gaussian shape of LLM weights — info-theoretically optimal for that distribution. |
| **Double quantization** | Quantize the *quantization constants* (the per-block fp32 scales) too. Saves ~0.4 bits/param — about 0.3 GB on a 7B model — at near-zero accuracy cost. |
| **Paged optimizers** | Optimizer state lives in CPU/unified memory and is paged to GPU on demand (NVIDIA unified memory), so gradient-checkpointing memory spikes don't OOM. |
| **Compute dtype** | Frozen weights are stored NF4 but **dequantized to bf16 on the fly** for each matmul. You store 4-bit, you compute in 16-bit. |
| **Target modules** | Which layers get adapters. Attention projections (`q_proj`,`k_proj`,`v_proj`,`o_proj`) are standard; adding the MLP projections helps quality at more cost. |
| **Adapter merging** | After training you can fold `B·A` back into `W` (`merge_and_unload`) for adapter-free inference — but merging into a *de*quantized base, not the 4-bit one. |
| **PEFT** | Hugging Face's Parameter-Efficient Fine-Tuning library that implements LoRA/QLoRA on top of `transformers` + `bitsandbytes`. |

## 4. Setup

The worked examples below use **only NumPy and PyTorch (CPU)** so they run anywhere with no GPU and no model
download — they implement the NF4 quantization math and the LoRA adapter directly, which is where the
understanding lives. The real end-to-end recipe needs a CUDA GPU plus `bitsandbytes`, so it's **gated** behind
an `os.getenv` flag and shows the exact API shape.

```bash
# The real QLoRA stack (CUDA GPU required for bitsandbytes 4-bit):
pip install "transformers>=4.44" "peft>=0.11" "bitsandbytes>=0.43" accelerate datasets trl
```

`bitsandbytes` is effectively CUDA-only, which is why the executable cells here stay framework-light; the GPU
cell only fires if you set the flag on a CUDA box.

In [1]:
# Environment probe — what's available in THIS kernel (no GPU, no downloads needed).
import importlib.util
import sys

import numpy as np


def have(mod: str) -> str:
    return "installed" if importlib.util.find_spec(mod) else "not installed"


print(f"python        : {sys.version.split()[0]}")
print(f"numpy         : {np.__version__}")
for m in ("torch", "transformers", "peft", "bitsandbytes", "accelerate", "trl"):
    print(f"{m:<14}: {have(m)}")

print("\nExamples 1 & 2 below are pure NumPy/PyTorch (CPU) and run regardless of the above.")

python        : 3.13.7
numpy         : 2.5.0
torch         : installed
transformers  : installed
peft          : not installed
bitsandbytes  : not installed
accelerate    : not installed
trl           : not installed

Examples 1 & 2 below are pure NumPy/PyTorch (CPU) and run regardless of the above.


## 5. Worked Examples

### Example 1 — NF4 vs plain INT4: why the data type matters (the "Q")

QLoRA's headline accuracy comes partly from **NF4**, a 4-bit type whose 16 levels are placed at the quantiles
of a standard normal — exactly where Gaussian-distributed weights concentrate. Plain INT4 spaces its 16
levels *uniformly*, wasting resolution in the tails where almost no weights live. Here we quantize a Gaussian
weight vector both ways (per-block **absmax** scaling, like real NF4) and compare reconstruction error.

In [2]:
# The 16 NF4 levels (normalized to [-1, 1]), from the QLoRA paper / bitsandbytes.
NF4 = np.array(
    [-1.0, -0.6961928, -0.52507305, -0.39491749, -0.28444138, -0.18477343,
     -0.09105004, 0.0, 0.07958030, 0.16093020, 0.24611230, 0.33791524,
     0.44070983, 0.56261700, 0.72295684, 1.0],
    dtype=np.float32,
)

rng = np.random.default_rng(0)
# Stand-in for one weight block: ~Gaussian, as LLM weights tend to be.
W = rng.standard_normal(4096).astype(np.float32) * 0.05


def quant_nf4(w):
    """Absmax-scale to [-1,1], snap each value to the nearest NF4 quantile, rescale."""
    scale = np.abs(w).max()                       # one absmax scale per block
    idx = np.abs((w / scale)[:, None] - NF4[None, :]).argmin(axis=1)
    return NF4[idx] * scale


def quant_int4(w):
    """Uniform signed 4-bit grid (levels evenly spaced)."""
    qmax = 7
    scale = np.abs(w).max() / qmax
    q = np.clip(np.round(w / scale), -8, 7)
    return q * scale


def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))


print(f"plain INT4 (uniform levels) : RMSE = {rmse(W, quant_int4(W)):.6f}")
print(f"NF4 (normal-quantile levels): RMSE = {rmse(W, quant_nf4(W)):.6f}")
print(f"\nNF4 is {rmse(W, quant_int4(W)) / rmse(W, quant_nf4(W)):.2f}x lower error on Gaussian "
      "weights — same 4 bits, levels placed where the mass is.")

plain INT4 (uniform levels) : RMSE = 0.008091
NF4 (normal-quantile levels): RMSE = 0.005530

NF4 is 1.46x lower error on Gaussian weights — same 4 bits, levels placed where the mass is.


### Example 2 — a frozen base + a trainable LoRA adapter (the "LoRA")

This is the QLoRA training recipe in miniature, in pure PyTorch on CPU: a **frozen** linear layer (stand-in
for the 4-bit base) plus a low-rank adapter `ΔW = (α/r)·B·A` that *is* trainable. We count parameters (the
adapter is a few percent), run a few optimizer steps, and confirm the **base never changes** while the output
adapts — the whole point of QLoRA.

In [3]:
import torch
import torch.nn as nn

torch.manual_seed(0)
d_in, d_out, r, alpha = 512, 512, 8, 16

# Frozen base — the 4-bit-quantized weights you never update.
base = nn.Linear(d_in, d_out, bias=False)
base.requires_grad_(False)

# LoRA adapter: A random (down-project), B zero (up-project) -> ΔW starts at 0.
A = nn.Parameter(torch.randn(r, d_in) * 0.01)
B = nn.Parameter(torch.zeros(d_out, r))
scaling = alpha / r


def forward(x):
    return base(x) + scaling * (x @ A.t() @ B.t())


base_params = sum(p.numel() for p in base.parameters())
lora_params = A.numel() + B.numel()
print(f"frozen base params : {base_params:,}")
print(f"trainable LoRA     : {lora_params:,}  ({lora_params / base_params:.1%} of base)")

# One tiny "fine-tuning" run toward a random target.
x, target = torch.randn(8, d_in), torch.randn(8, d_out)
base_snapshot = base.weight.detach().clone()
opt = torch.optim.Adam([A, B], lr=1e-2)
for step in range(100):
    opt.zero_grad()
    loss = ((forward(x) - target) ** 2).mean()
    loss.backward()
    opt.step()

print(f"\nloss 100 steps later : {loss.item():.4f}")
print(f"base weights changed : {not torch.equal(base.weight, base_snapshot)}  (LoRA only touches A, B)")
print(f"base.weight.grad     : {base.weight.grad}  (no gradient ever allocated for the frozen base)")

frozen base params : 262,144
trainable LoRA     : 8,192  (3.1% of base)



loss 100 steps later : 0.0000
base weights changed : False  (LoRA only touches A, B)
base.weight.grad     : None  (no gradient ever allocated for the frozen base)


### Example 3 — the real QLoRA stack (gated)

In practice you never hand-roll the above — you stack `bitsandbytes` (4-bit load) + `peft` (LoRA) +
`transformers`/`trl` (training). The cell below shows the exact, current API shape. It's **gated behind
`RUN_QLORA`** and a CUDA check so the notebook still executes top-to-bottom on a plain CPU; flip the env var
on a GPU box (with the libraries installed) to actually load a model.

In [4]:
import os

_torch = importlib.util.find_spec("torch")
GPU = bool(_torch) and __import__("torch").cuda.is_available()
HAVE_LIBS = all(importlib.util.find_spec(m) for m in ("transformers", "peft", "bitsandbytes"))

if os.getenv("RUN_QLORA") and GPU and HAVE_LIBS:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig

    # 1) Load the base in 4-bit NF4 with double quantization (this is the "Q").
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=__import__("torch").bfloat16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        "meta-llama/Llama-3.2-1B", quantization_config=bnb, device_map="auto"
    )
    model = prepare_model_for_kbit_training(model)  # casts norms to fp32, enables grad ckpt

    # 2) Attach LoRA adapters (this is the "LoRA").
    lora = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
    model = get_peft_model(model, lora)
    model.print_trainable_parameters()  # e.g. "trainable: 0.6% of all params"
    # 3) Train with Trainer/SFTTrainer using a paged optimizer: optim="paged_adamw_8bit".
else:
    print("Skipping GPU path (set RUN_QLORA=1 on a CUDA box with the libs). Recipe shape:\n")
    print(
        "# 1) 4-bit NF4 base (bitsandbytes):\n"
        "BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',\n"
        "                   bnb_4bit_use_double_quant=True,\n"
        "                   bnb_4bit_compute_dtype=torch.bfloat16)\n\n"
        "# 2) LoRA adapters (peft):\n"
        "LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type='CAUSAL_LM',\n"
        "           target_modules=['q_proj','k_proj','v_proj','o_proj'])\n"
        "model = get_peft_model(prepare_model_for_kbit_training(model), lora)\n\n"
        "# 3) Train with a paged optimizer so memory spikes don't OOM:\n"
        "TrainingArguments(optim='paged_adamw_8bit', gradient_checkpointing=True, ...)\n\n"
        "# 4) Save just the adapter (a few MB), or merge for adapter-free inference:\n"
        "model.save_pretrained('my-qlora-adapter')   # tiny\n"
        "model.merge_and_unload()                    # fold ΔW into a dequantized base"
    )

Skipping GPU path (set RUN_QLORA=1 on a CUDA box with the libs). Recipe shape:

# 1) 4-bit NF4 base (bitsandbytes):
BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                   bnb_4bit_use_double_quant=True,
                   bnb_4bit_compute_dtype=torch.bfloat16)

# 2) LoRA adapters (peft):
LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type='CAUSAL_LM',
           target_modules=['q_proj','k_proj','v_proj','o_proj'])
model = get_peft_model(prepare_model_for_kbit_training(model), lora)

# 3) Train with a paged optimizer so memory spikes don't OOM:
TrainingArguments(optim='paged_adamw_8bit', gradient_checkpointing=True, ...)

# 4) Save just the adapter (a few MB), or merge for adapter-free inference:
model.save_pretrained('my-qlora-adapter')   # tiny
model.merge_and_unload()                    # fold ΔW into a dequantized base


## 6. Gotchas & Pitfalls

- **You can't merge adapters into a 4-bit base.** `merge_and_unload()` needs to add `ΔW` to real weights, so
  merging happens against a **dequantized** copy. Plan VRAM for that, or keep the adapter separate and load it
  on top of the 4-bit base at inference (the lighter path).
- **Quantization noise ≠ training instability — but precision of the adapter matters.** Train adapters and
  layer norms in bf16/fp32. `prepare_model_for_kbit_training` handles casting norms to fp32 and enabling
  gradient checkpointing; skipping it causes NaNs or silent quality loss.
- **`target_modules` is a real quality lever.** Attention-only adapters are the cheap default; adding MLP
  projections (`gate_proj`,`up_proj`,`down_proj`) often helps but raises memory and trainable count. Don't
  forget to include them when a task underfits.
- **Rank/alpha confusion.** The effective adapter strength is `alpha/r`. If you bump `r` and keep `alpha`
  fixed, you've quietly *shrunk* the scaling. A common convention is `alpha = 2·r`.
- **4-bit is for the frozen base only.** Gradients flow through the dequantized weights to the adapters; the
  4-bit weights themselves never receive updates. QLoRA is a fine-tuning *of adapters*, not of the base.
- **bitsandbytes is CUDA-first.** There's no production CPU path; you can't actually run the 4-bit base on a
  Mac/CPU. For CPU inference of a finished model, merge and export to GGUF (see the GGUF notebook).
- **Paged optimizer ≠ free lunch.** It prevents OOM spikes by paging optimizer state to host memory, but if
  you page constantly (too-large batch) throughput tanks. It's a safety valve, not a capacity multiplier.
- **Save the adapter, not the base.** A trained QLoRA adapter is a few MB. Don't re-upload the multi-GB base;
  ship the adapter and point it at the public base model.
- **Inference dtype mismatch.** Loading an adapter trained with `compute_dtype=bf16` onto an fp16 base can
  shift outputs slightly; keep dtypes consistent between train and serve.

## 7. When to Use vs Alternatives

| Option | Trainable params | Base memory | Best for | Trade-off |
|--------|------------------|-------------|----------|-----------|
| **QLoRA** (4-bit base + LoRA) | ~0.1–1% | 4-bit (~0.5 B/param) | Fine-tuning big models on **one** GPU | Slightly slower step (dequant on the fly); can't train base |
| **LoRA** (fp16 base + LoRA) | ~0.1–1% | fp16 (~2 B/param) | Fine-tuning when the fp16 base already fits | 4× the base memory of QLoRA |
| **Full fine-tuning** | 100% | fp16 + grads + Adam (~16 B/param) | Max quality, you have the hardware | Enormous memory; easy to overfit small data |
| **Prompt / prefix tuning** | tiny | fp16 base | Cheapest adaptation, many tasks share a base | Lower ceiling than LoRA on hard tasks |
| **Quantize-only (GPTQ/AWQ/bnb)** | 0 (no training) | 4-bit | **Inference** of an already-good model | Doesn't adapt behavior at all |
| **Unsloth / Axolotl** | same as QLoRA | 4-bit | QLoRA but **2–5× faster / less VRAM** via fused kernels | Another dependency; see the Unsloth notebook |

**Rules of thumb:**
- Want to fine-tune and it doesn't fit for full training → **QLoRA** (the default answer in 2024+).
- It *does* fit in fp16 and you want a touch more quality/speed → plain **LoRA** (skip the quant overhead).
- You only need to *serve* an existing model smaller/faster → **quantize-only**, no adapters (Quantization nb).
- Doing lots of QLoRA runs and want them faster → **Unsloth/Axolotl** wrap the same recipe with speedups.
- Tiny task, many variants off one base → consider **prompt/prefix tuning** before reaching for LoRA.

## 8. Resources

- **QLoRA paper** — *Dettmers, Pagnoni, Holtzman, Zettlemoyer, "QLoRA: Efficient Finetuning of Quantized LLMs"* (NeurIPS 2023), introduces NF4, double quantization, and paged optimizers: <https://arxiv.org/abs/2305.14314>
- **LoRA paper** — *Hu et al., "LoRA: Low-Rank Adaptation of Large Language Models"*: <https://arxiv.org/abs/2106.09685>
- **PEFT docs** — LoRA/QLoRA config, `get_peft_model`, target modules, merging: <https://huggingface.co/docs/peft>
- **Transformers — Quantization (bitsandbytes 4-bit / NF4)**: <https://huggingface.co/docs/transformers/main/en/quantization/bitsandbytes>
- **bitsandbytes docs** — `BitsAndBytesConfig`, double quant, paged optimizers: <https://huggingface.co/docs/bitsandbytes>
- **TRL `SFTTrainer`** — the usual training loop driver for QLoRA: <https://huggingface.co/docs/trl/sft_trainer>

**Cross-refs in this library:** [Quantization: GPTQ/AWQ/bitsandbytes](./quantization-gptq-awq.ipynb) (the NF4 / 4-bit machinery), [LoRA / ControlNet](./lora-controlnet.ipynb) (the low-rank adapter idea), [Unsloth](./unsloth.ipynb) (faster QLoRA), [GGUF & llama.cpp](./gguf-llama-cpp.ipynb) (export a merged model for CPU inference).